[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NguyenVu04/band-tilt/blob/main/notebooks/01_clean_and_split.ipynb)

# 01 — Cleaning and Splitting

**Purpose.** Turn the raw MDT export into a clean, frozen train/test partition.
PROJECT.md section 19 Steps 2 and 3.

**Inputs.** `data/raw/` — read-only.

**Outputs.** `data/processed/mdt_train.parquet` and `mdt_test.parquet`.

### The boundary rule — the most important rule in this project

Cleaning may drop records that violate an **externally known** bound: a standard,
a specification, a physical limit. It may not apply anything **learned from the
data** — no quantile clip, no IQR fence, no z-score cut.

The reason is that this notebook sees the whole dataset, test records included.
A threshold computed here has been informed by the test data, so every downstream
result is measured against records that helped decide what counts as valid. That
failure is invisible: nothing raises, and the numbers simply come out better than
they should.

Statistical outlier handling, if it is needed at all, belongs in notebook 04,
fitted on the training partition only.

## 0. Environment

Run this section first, wherever you are.

**Locally** it only walks up to the project root and makes it the working
directory, so the root-relative paths in `configs/data.yaml` resolve the same way
they do for `task clean:data` and the DVC pipeline. Nothing is installed.

**In Colab** it also clones the repository, puts it on `sys.path` so `import src`
works without an editable install, and installs the packages Colab does not ship.
Note that `data/` and `models/` are DVC-tracked and therefore *not* part of the
clone — a fresh runtime has neither. See the Drive cell below.

In [ ]:
# --- Environment bootstrap -------------------------------------------------
# Identical in every notebook. Forked the repository? Change these three values
# and the badge URL at the top of this notebook.
REPO_URL = "https://github.com/NguyenVu04/band-tilt.git"
BRANCH = "main"
SUBDIR = ""  # the project root is the repository root

# (import name, pip name). Colab already ships numpy, pandas, pyarrow,
# scikit-learn, joblib, matplotlib and seaborn, so only these are installed —
# which keeps the bootstrap fast and avoids a "restart runtime" prompt.
COLAB_PACKAGES = [("hydra", "hydra-core")]

import importlib.util
import os
import subprocess
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    checkout = Path("/content") / Path(REPO_URL).stem
    if not checkout.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(checkout)],
            check=True,
        )
    root = checkout / SUBDIR
    missing = [pip for mod, pip in COLAB_PACKAGES if importlib.util.find_spec(mod) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
else:
    # JupyterLab starts the kernel in notebooks/; walk up to the project root.
    root = Path.cwd()
    while not (root / "pyproject.toml").exists() and root != root.parent:
        root = root.parent

os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))  # makes `import src` work without an editable install

# Extra Hydra overrides consumed by load_config() in section 1. Empty unless
# the Drive cell below fills it in, so local runs are unaffected.
CONFIG_OVERRIDES: list[str] = []

print(f"project root: {root}   colab: {IN_COLAB}")

In [ ]:
# --- Colab: data and artifacts (optional) ----------------------------------
# data/ and models/ are DVC-tracked, so they are not in the Git clone and a
# fresh Colab runtime has neither. Mount Drive and point the config at it —
# Drive also survives a runtime reset, which /content does not.
#
# The scene is the large one: data/external/simulation_map/ holds 3,753 meshes,
# so keep it on Drive rather than re-downloading it per session.
#
# from google.colab import drive
#
# drive.mount("/content/drive")
# DATA_ROOT = "/content/drive/MyDrive/band-tilt/data"
# CONFIG_OVERRIDES += [
#     f"data.mdt_path={DATA_ROOT}/raw/measurement_data.csv",
#     f"data.cell_config_path={DATA_ROOT}/raw/gcell_conf.csv",
#     f"data.scene_file={DATA_ROOT}/external/simulation_map/scene.xml",
#     f"data.train_path={DATA_ROOT}/processed/mdt_train.parquet",
#     f"data.test_path={DATA_ROOT}/processed/mdt_test.parquet",
# ]

## 1. Setup

Compose the config, seed everything, and import from `src/`. Every notebook
starts the same way so that a cell copied between notebooks behaves identically.

In [ ]:
# Standard setup for every notebook in this project.
# Autoreload so edits in src/ take effect without restarting the kernel.
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src.config import load_config
from src.utils.plotting import setup_plotting
from src.utils.seed import set_seed

cfg = load_config(overrides=CONFIG_OVERRIDES)
set_seed(cfg.seed)
setup_plotting()  # matplotlib/seaborn styling for report-ready figures

pd.set_option("display.max_columns", 50)
cfg

## 2. Load the raw inputs

In [ ]:
from src.data import clean, schema, split
from src.data.load import load_cell_config, load_mdt, save_processed

mdt = load_mdt(cfg)
cells = load_cell_config(cfg)
audit = [("raw", len(mdt))]
print(f"{len(mdt):,} raw MDT records")

## 3. Validate against the declared schema

Non-strict first: collect every violation and look at them together, rather than
stopping at the first one. Strict validation runs again in section 9, after
cleaning, which is what proves the cleaning actually worked.

The cell configuration is validated non-strict permanently — the multi-band
columns are declared but not yet exported (docs/adr/0005).

In [ ]:
mdt = schema.validate(mdt, cfg, contract="mdt", strict=False)
cells = schema.validate(cells, cfg, contract="cell_config", strict=False)

schema.find_violations(mdt, cfg, contract="mdt").groupby(["column", "rule"]).size()

## 4. Remove duplicate records

On the full record, not on `ue_id`. One UE contributing many measurements is what
an MDT export *is*, so deduplicating by identifier would delete most of the
dataset.

In [ ]:
mdt = clean.drop_duplicates(mdt, cfg)
audit.append(("duplicates removed", len(mdt)))
print(f"{len(mdt):,} records remain")

## 5. Drop unreportable RSRP

Outside the 3GPP TS 38.133 range `[-156, -31]` dBm. The raw export contains
`rsrp = 0.0`, which is not a measurable RSRP.

Watch the count. A sentinel appearing in a large fraction of records is an
upstream export problem, not something to quietly filter away.

In [ ]:
mdt = clean.drop_invalid_rsrp(mdt, cfg)
audit.append(("invalid RSRP removed", len(mdt)))
print(f"{len(mdt):,} records remain")

## 6. Drop measurements against unconfigured cells

A cell with no position, azimuth or tilt cannot be placed in the Sionna-RT scene,
so a measurement against it can never be compared to a simulated radio map.

In [ ]:
mdt = clean.drop_unknown_cells(mdt, cells, cfg)
audit.append(("unknown cells removed", len(mdt)))
print(f"{len(mdt):,} records remain")

## 7. Drop records outside the scene

The evaluation grid covers the scene, so a record outside it has no grid cell to
contribute to. Bounds come from `src.radio.scene.scene_bounds` — the same source
the radio map and the UE density grid use, which is what keeps all three
aligned.

In [ ]:
mdt = clean.drop_outside_scene(mdt, cfg)
audit.append(("outside scene removed", len(mdt)))
print(f"{len(mdt):,} records remain")

## 8. Cleaning audit

The record of what was dropped and why. This table is the evidence that the
filters did what was intended — and the place where a rule that removes far more
than expected becomes visible.

In [ ]:
audit_df = pd.DataFrame(audit, columns=["step", "records"])
audit_df["removed"] = -audit_df["records"].diff().fillna(0).astype(int)
audit_df["cumulative_pct"] = 100 * (1 - audit_df["records"] / audit_df["records"].iloc[0])
audit_df

## 9. Re-validate, strictly

The cleaned frame must satisfy the contract completely. A violation surviving to
this point means a cleaning rule did not cover a case the schema declares.

In [ ]:
mdt = schema.validate(mdt, cfg, contract="mdt", strict=True)
print(f"{len(mdt):,} records pass strict validation")

## 10. Leakage-safe split

`cfg.data.split.method` decides which kind of generalisation the final estimate
measures — see docs/adr/0007. Whichever is configured, this is the only place in
the project that splits data.

**The test partition is frozen from here until notebook 06.**

In [ ]:
train_df, test_df = split.train_test_split(mdt, cfg)
print(f"train: {len(train_df):,}   test: {len(test_df):,}")
print(f"test fraction: {len(test_df) / len(mdt):.1%} (configured {cfg.data.split.test_size:.0%})")

## 11. Leakage check

Cheap to run, and it fails loudly rather than producing an optimistic number.
Run it even when the splitter already asserts internally.

In [ ]:
split.assert_no_leakage(train_df, test_df, cfg)

comparison = pd.DataFrame(
    {"train": train_df["rsrp"].describe(), "test": test_df["rsrp"].describe()}
)
# The two distributions should be similar. A large difference is not necessarily
# a bug — a spatial split holds out a genuinely different area — but it must be
# understood before any result is reported.
comparison

## 12. Persist and version

Parquet, so dtypes round-trip. Then `dvc add` the outputs and commit the `.dvc`
files: the split is only frozen if it can be restored exactly.

In [ ]:
save_processed(train_df, cfg.data.train_path)
save_processed(test_df, cfg.data.test_path)
print(f"wrote {cfg.data.train_path} and {cfg.data.test_path}")

## 13. Handoff checklist

- [ ] Every cleaning rule traces to an external source recorded in `configs/data.yaml`.
- [ ] The audit table in section 8 accounts for every dropped record.
- [ ] Nothing in this notebook computed a threshold from the data.
- [ ] The leakage check in section 11 passed.
- [ ] Both parquet files are written and DVC-tracked.
- [ ] `configs/data.yaml` matches what was actually run.

**The test split is now frozen.** It is not looked at again until notebook 06.